# SPARQL — User Guide

Two things in one guide, since they're both "SPARQL" but otherwise fairly separate: querying RDF 1.2 data (triple-term-aware functions, quoted-triple patterns, annotation shorthand in queries), and treating a *query itself* as RDF data you can encode, inspect, edit, and validate.

See the [Graphs guide](02-graphs.ipynb) for the `TripleTerm`/`DirLangString`/reification semantics the query examples below build on, and [SHACL rules](04b-shacl-inference-rules.ipynb) for `sh:sparql`/`sh:SPARQLRule`, which use these same query functions inside SHACL shapes.

A SPARQL-RL (SRL) Datalog-style rules layer is a separate, deliberately out-of-scope idea - see [3.a](03a-sparql-rules-pending.md).

## How to run this notebook

1. `pip install "git+https://github.com/hidden-graph/starlayer.git"` (or install the three packages editable from a local checkout — see the root [README](../../README.md)).
2. Run cells from top to bottom — later sections reuse variables from earlier ones.

In [1]:
from starlayergraph import StarLayerGraph, Namespace, TripleTerm, DirLangString, Literal
from starshacl import StarShaclValidator

EX = Namespace("http://example.org/")
RDF = Namespace("http://www.w3.org/1999/02/22-rdf-syntax-ns#")

## 1. Query semantics and built-in functions

Supports SPARQL 1.2 query execution:
- Query over reified quoted triples
- Turtle 1.2 quoted-triple syntax in queries (`<<( ... )>>`)
- Binds variables for terms inside triple terms (`?s ?p ?o`)

The examples below run against one running example graph, parsed here (the same document used in the [Graphs guide](02-graphs.ipynb)'s parsing section).

In [2]:
g_parsed = StarLayerGraph()
g_parsed.bind("ex", EX)
g_parsed.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

    # language-direction literals
    ex:note_en ex:text "hello"@en--ltr .
    ex:note_ar ex:text "مرحبا"@ar--rtl .

    # canonical reification with rdf:reifies
    ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> ;
      ex:source ex:wikipedia ;
      ex:confidence "high" .

    # anonymous inline annotation block
    ex:bob ex:likes ex:dana {| ex:since "2020" ; ex:source ex:LinkedIn |} .

    # named reifier with annotations
    ex:bob ex:mentions ex:erin ~ ex:stmt1 {| ex:confidence "0.9" ; ex:source ex:WikiData |} .

    # named reifier without annotation block
    ex:bob ex:worksWith ex:frank ~ ex:stmt2 .

    # an additional quoted triple term reused in the query examples below
    ex:alice ex:mentions <<( ex:bob ex:likes ex:dana )>> .
''', format='turtle12')
print("parsed triples:", len(g_parsed))

parsed triples: 16


In [3]:
# SPARQL query to bind all terms inside a quoted triple
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?claim ?s ?p ?o ?source WHERE {
  ?claim rdf:reifies <<( ?s ?p ?o )>> .
  ?claim ex:source ?source .
  FILTER(?p = ex:knows)
}
ORDER BY ?claim ?s ?o
""")

for row in rows:
    print(
        g_parsed.qname(row.claim),
        g_parsed.qname(row.s),
        g_parsed.qname(row.p),
        g_parsed.qname(row.o),
        "Source: ", g_parsed.qname(row.source),
    )

ex:claim ex:bob ex:knows ex:carol Source:  ex:wikipedia


In [4]:
# detect triple-term values using isTRIPLE
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?claim ?p WHERE {
  ?claim ?p ?statement .
  FILTER( isTRIPLE(?statement) )
}
ORDER BY ?claim
""")

for row in rows:
    print(g_parsed.qname(row.claim), g_parsed.qname(row.p))

ex:alice ex:mentions
ex:claim rdf:reifies
ex:stmt1 rdf:reifies
ex:stmt2 rdf:reifies
rr:0 rdf:reifies


### Additional SPARQL 1.2 functions

- `TRIPLE(s, p, o)` — a functional way to write a triple term `<<( s p o )>>`
- `SUBJECT()`, `PREDICATE()`, `OBJECT()` — extract the parts out of a triple term
- `LANGDIR()`, `hasLANGDIR()`, `STRLANGDIR()` — read, check, and build a literal's base direction
- `LANG()`, `hasLANG()` — SPARQL 1.1 functions, now also aware of direction-tagged literals

In [5]:
# TRIPLE() and SUBJECT()/PREDICATE()/OBJECT()
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?s ?p ?o WHERE {
  ex:claim rdf:reifies ?t .
  FILTER(?t = TRIPLE(ex:bob, ex:knows, ex:carol))
  BIND(SUBJECT(?t) AS ?s)
  BIND(PREDICATE(?t) AS ?p)
  BIND(OBJECT(?t) AS ?o)
}
""")
for row in rows:
    print(g_parsed.qname(row.s), g_parsed.qname(row.p), g_parsed.qname(row.o))

ex:bob ex:knows ex:carol


In [6]:
# LANGDIR() / hasLANGDIR() / LANG() / hasLANG() over the direction-tagged notes
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
SELECT ?s ?lang ?dir ?hasDir WHERE {
  ?s ex:text ?lit .
  BIND(LANG(?lit) AS ?lang)
  BIND(LANGDIR(?lit) AS ?dir)
  BIND(hasLANGDIR(?lit) AS ?hasDir)
}
ORDER BY ?s
""")
for row in rows:
    print(g_parsed.qname(row.s), row.lang, row.dir, row.hasDir)

# STRLANGDIR() constructs a direction-tagged literal directly from plain strings
rows = g_parsed.query('SELECT ?lit WHERE { BIND(STRLANGDIR("hi", "en", "ltr") AS ?lit) }')
for row in rows:
    print(row.lit.n3())

ex:note_ar ar rtl true
ex:note_en en ltr true
"hi"@en--ltr


### Turtle annotation shorthand inside SPARQL queries

The `{| ?pred ?val |}`, `~ ?r`, and `<< s p o >>` Turtle syntax used to parse and serialize graphs also works directly inside a SPARQL 1.2 `WHERE` clause — querying using the same shorthand a graph is serialized with, without expanding to `rdf:reifies`/`<<( )>>` by hand.

In [7]:
# {| ?pred ?val |}: query an anonymous reifier's annotations inline
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?pred ?val WHERE {
  ex:bob ex:likes ex:dana {| ?pred ?val |}
  FILTER(?pred != rdf:reifies)
}
ORDER BY ?pred
""")
for row in rows:
    print(g_parsed.qname(row.pred), row.val)

# ~ ?r: bind the reifier itself for a named reifier
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
SELECT ?r WHERE {
  ex:bob ex:worksWith ex:frank ~ ?r
}
""")
for row in rows:
    print(g_parsed.qname(row.r))

# << s p o >> ?pred ?val: reification shorthand, no assertion required
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?pred ?val WHERE {
  << ex:bob ex:likes ex:dana >> ?pred ?val
  FILTER(?pred != rdf:reifies)
}
ORDER BY ?pred
""")
for row in rows:
    print(g_parsed.qname(row.pred), row.val)

ex:since 2020
ex:source http://example.org/LinkedIn
ex:stmt2
ex:since 2020
ex:source http://example.org/LinkedIn


## 2. Encoding, editing, and validating a query as RDF

`starsparql` represents a SPARQL query itself as an RDF graph, based on SPARQL 1.2 algebra, using the `salg:` ontology (specific to starlayer). Queries can be exported to RDF and imported back. A corresponding SHACL shapes graph validates the encoding; `starshacl` separately extends the SHACL meta-shape graph for SHACL 1.2 semantics.

Grounded in one tiny graph and one tiny query throughout, so each step's effect shows up as a real, different query result — not just a triple count. Eight steps: build a graph + query → parse the query into a `Query` object → encode its algebra as RDF → edit the query by editing the RDF → decode back into a real `Query` and run it → export the edited query back to text → load the ontology/shapes graphs → validate an encoded query with RDFS reasoning.

### Step 1 — a graph and a query

Everything below is grounded in this one tiny graph and one tiny query, so later steps' effects show up as real, different query results — not just triple counts.

In [8]:
import starsparql

g = StarLayerGraph()
g.bind("ex", EX)
g.add((EX.bob, EX.knows, EX.carol))
g.add((EX.bob, EX.likes, EX.dana))

text_query = """PREFIX ex: <http://example.org/>
SELECT ?s ?o WHERE { ?s ex:knows ?o }"""

for row in g.query(text_query):
    print(g.qname(row.s), g.qname(row.o))

ex:bob ex:carol


### Step 2 — parse the query text into a `Query` object

`prepare_query_12()` parses text into a real, executable-shaped `rdflib.Query` — the SPARQL-1.2-aware counterpart to plain rdflib's own `prepareQuery()`. Its grammar is a strict superset of SPARQL 1.1, so it's the right function to call here even though this particular query uses no RDF 1.2 syntax at all.

In [9]:
parsed = starsparql.prepare_query_12(text_query)

print(type(parsed))
print("top-level algebra op:", parsed.algebra.name)

<class 'rdflib.plugins.sparql.sparql.Query'>
top-level algebra op: SelectQuery


### Step 3 — encode the algebra as RDF

This is the actual "get access to the RDF" step. `query_to_rdf()` walks `parsed.algebra` and encodes it into an `rdflib.Graph` using the `salg:` ontology — one triple pattern, filter, projection, etc. per algebra node. `root` is the graph node standing in for the query's top-level operator, needed to decode the graph back into a `Query` later.

In [10]:
graph, root = starsparql.query_to_rdf(parsed)
print("encoded triples:", len(graph))

encoded triples: 143


### Step 4 — edit the query by editing the RDF

Rewrite the triple pattern's predicate directly in the RDF graph — `ex:knows` → `ex:likes`. No text-level find/replace anywhere; this edits the query's own structure.

In [11]:
from starsparql.vocab import SALG

for s, p, o in list(graph.triples((None, SALG.predicate, EX.knows))):
    graph.remove((s, p, o))
    graph.add((s, p, EX.likes))

### Step 5 — decode back into a real `Query`, and run it

`rdf_to_query()` turns the edited RDF back into an executable `Query`. Running it against the *same* graph from Step 1 proves the edit is semantic — the result set genuinely changes, from `(bob, carol)` to `(bob, dana)`.

In [12]:
modified_query = starsparql.rdf_to_query(graph, root)

for row in g.query(modified_query):
    print(g.qname(row.s), g.qname(row.o))

ex:bob ex:dana


### Step 6 — export the modified query back to text

`translate_algebra_12()` walks a `Query`'s `.algebra` straight to SPARQL 1.2 text — no intermediate RDF round-trip needed when text is all you want.

In [13]:
print(starsparql.translate_algebra_12(modified_query))

SELECT ?s ?o{?s <http://example.org/likes> ?o. }


### Step 7 — load the ontology and shapes graphs

`starsparql.ontology_graph()` and `starsparql.shapes_graph()` each return a fresh **plain `rdflib.Graph`**, not a `StarLayerGraph` — confirmed from source (`starsparql/ontology/__init__.py`, `starsparql/ontology/sparql_shapes.py`). They contain the `salg:`/SHACL *vocabulary itself* (classes, properties, shape definitions) parsed with plain `format="turtle"`, never actual triple-term or reification data, so there's nothing RDF-1.2-specific for them to carry — a plain `rdflib.Graph` is the right and complete return type here. `shapes_graph()` additionally requires `pyshacl` to be installed.

In [14]:
ontology = starsparql.ontology_graph()
print("ontology triples:", len(ontology))

shapes = starsparql.shapes_graph()
print("shapes triples:", len(shapes))

ontology triples: 333
shapes triples: 1265


### Step 8 — validate an encoded query, with RDFS reasoning over the ontology

`starsparql.validate()` runs the shapes graph against a data graph with `inference="rdfs"` and `ont_graph=ontology_graph()` — real RDFS reasoning before SHACL validation, which is what lets shapes like `GraphPatternShape`/`ExpressionShape` check a single abstract superclass (`salg:Expression`) instead of enumerating every concrete operator/builtin by name. This example builds its query with plain rdflib's own `prepareQuery()` (not `starsparql.prepare_query_12()`) deliberately, to show `validate()` works on any correctly-encoded `salg:` graph, regardless of what produced it.

In [15]:
from rdflib.plugins.sparql import prepareQuery

q = prepareQuery("SELECT ?s ?o WHERE { ?s ex:knows ?o }", initNs={"ex": EX})
encoded, root = starsparql.query_to_rdf(q)
print("encoded query triples:", len(encoded))

conforms, results_graph, results_text = starsparql.validate(encoded)
print("conforms:", conforms)

encoded query triples: 143


conforms: True


## Further work

- **SPARQL-RL (SRL) rules** are deliberately out of scope for this project — see [3.a](03a-sparql-rules-pending.md).
- **`sh:sparql` and `sh:SPARQLRule`** use these same query functions from inside SHACL shapes — see the [SHACL inference rules guide](04b-shacl-inference-rules.ipynb).